# pems11 按年节点增长数据处理

规则：
- 扫描 pems11 目录下 `d11_text_station_5min_YYYY_MM_DD.txt.gz`。
- **首年**：以数据集**首日**的 stations 作为该年的固定节点集合。
- **其余年份**：以每年 **1月1日** 当天的 stations 作为该年的节点集合。
- 每年只保留「该年锚定日」出现的节点，每年保存为一个 CSV。

In [2]:
import os
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from collections import defaultdict

In [3]:
PEMS_DIR = './pems11'
OUTPUT_DIR = '.'
PREFIX = 'd11_text_station_5min'
SUFFIX = '.txt.gz'

COLUMN_NAMES = [
    "Timestamp", "Station", "District", "Freeway #",
    "Direction of Travel", "Lane Type", "Station Length",
    "Samples", "% Observed", "Total Flow", "Avg Occupancy", "Avg Speed"
]

## 1. 读取 pems11 目录，解析所有可用日期

In [4]:
def list_pems_dates(data_dir):
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"目录不存在: {data_dir}")
    dates = {}
    for f in os.listdir(data_dir):
        if not f.startswith(PREFIX) or not f.endswith(SUFFIX):
            continue
        try:
            rest = f[len(PREFIX):-len(SUFFIX)].strip(' _')
            parts = rest.split('_')
            if len(parts) == 3:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                dt = date(y, m, d)
                dates[dt] = os.path.join(data_dir, f)
        except (ValueError, IndexError):
            continue
    return dates

available = list_pems_dates(PEMS_DIR)
sorted_dates = sorted(available.keys())
first_date = min(sorted_dates)
print(f"pems11 共 {len(available)} 个日期文件")
print(f"首日: {first_date}, 末日: {sorted_dates[-1]}")

FileNotFoundError: 目录不存在: ./pems11

## 2. 确定每年的锚定日与节点集合

In [4]:
def get_anchor_date(year, first_date):
    """首年用数据集首日，其余年份用该年 1 月 1 日。"""
    if year == first_date.year:
        return first_date
    return date(year, 1, 1)

def read_station_ids_from_file(path):
    df = pd.read_csv(path, header=None, usecols=[1], names=['Station'], compression='gzip')
    return np.sort(df['Station'].dropna().unique().astype(int))

years = sorted(set(d.year for d in sorted_dates))
year_anchor = {}
year_nodes = {}

for y in years:
    anchor = get_anchor_date(y, first_date)
    year_anchor[y] = anchor
    if anchor not in available:
        print(f"警告: {y} 年锚定日 {anchor} 无数据文件，跳过该年")
        continue
    path = available[anchor]
    year_nodes[y] = read_station_ids_from_file(path)
    print(f"{y} 年 锚定日={anchor}, 节点数={len(year_nodes[y])}")

1999 年 锚定日=1999-07-01, 节点数=424
2000 年 锚定日=2000-01-01, 节点数=556
2001 年 锚定日=2001-01-01, 节点数=622
2002 年 锚定日=2002-01-01, 节点数=631
2003 年 锚定日=2003-01-01, 节点数=646
2004 年 锚定日=2004-01-01, 节点数=646
2005 年 锚定日=2005-01-01, 节点数=838
2006 年 锚定日=2006-01-01, 节点数=944
2007 年 锚定日=2007-01-01, 节点数=1065
2008 年 锚定日=2008-01-01, 节点数=1098
2009 年 锚定日=2009-01-01, 节点数=1136
2010 年 锚定日=2010-01-01, 节点数=1193
2011 年 锚定日=2011-01-01, 节点数=1221
2012 年 锚定日=2012-01-01, 节点数=1273
2013 年 锚定日=2013-01-01, 节点数=1284
2014 年 锚定日=2014-01-01, 节点数=1390
2015 年 锚定日=2015-01-01, 节点数=1479
2016 年 锚定日=2016-01-01, 节点数=1424
2017 年 锚定日=2017-01-01, 节点数=1503
2018 年 锚定日=2018-01-01, 节点数=1505
2019 年 锚定日=2019-01-01, 节点数=1502
2020 年 锚定日=2020-01-01, 节点数=1488
2021 年 锚定日=2021-01-01, 节点数=1378
2022 年 锚定日=2022-01-01, 节点数=1378
2023 年 锚定日=2023-01-01, 节点数=1440
2024 年 锚定日=2024-01-01, 节点数=1440
2025 年 锚定日=2025-01-01, 节点数=1440


## 3. 按年汇总：只保留锚定日节点，每年保存一个 CSV

In [ ]:
def read_one_day(path, columns=None):
    if columns is None:
        columns = list(range(12))
    df = pd.read_csv(path, header=None, usecols=columns,
                    names=[COLUMN_NAMES[i] for i in columns], compression='gzip')
    return df


def generate_empty_day_df(day, node_set):
    """当某天文件缺失时，为该天所有节点生成 5 分钟分辨率、全 0 的占位数据。"""
    # PeMS 5min: 24*60/5 = 288 个时间点
    start_dt = datetime(day.year, day.month, day.day, 0, 0)
    timestamps = [
        (start_dt + timedelta(minutes=5 * i)).strftime("%m/%d/%Y %H:%M:%S")
        for i in range(288)
    ]

    rows = []
    for ts in timestamps:
        for st in node_set:
            rows.append([
                ts,          # Timestamp
                st,          # Station
                np.nan,      # District
                np.nan,      # Freeway #
                np.nan,      # Direction of Travel
                np.nan,      # Lane Type
                np.nan,      # Station Length
                0,           # Samples
                0,           # % Observed
                0,           # Total Flow
                0,           # Avg Occupancy
                0,           # Avg Speed
            ])

    return pd.DataFrame(rows, columns=COLUMN_NAMES)


os.makedirs(OUTPUT_DIR, exist_ok=True)

for year in years:
    if year not in year_nodes:
        continue
    node_set = set(year_nodes[year])
    start = first_date if year == first_date.year else date(year, 1, 1)
    end = date(year, 12, 31)
    chunks = []
    current = start
    while current <= end:
        if current in available:
            path = available[current]
            df = read_one_day(path)
            df = df[df['Station'].isin(node_set)]
        else:
            # 该天文件缺失，补 0
            df = generate_empty_day_df(current, node_set)
        chunks.append(df)
        current += timedelta(days=1)
    if not chunks:
        print(f"{year} 年无可用数据，跳过")
        continue
    out_df = pd.concat(chunks, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, f'pems11_{year}.csv')
    out_df.to_csv(out_path, index=False)
    print(f"{year}: 节点数={len(node_set)}, 行数={len(out_df)}, 已保存 -> {out_path}")

1999: 节点数=424, 行数=22463520, 已保存 -> .\pems11_1999.csv
2000: 节点数=556, 行数=58592352, 已保存 -> .\pems11_2000.csv
2001: 节点数=622, 行数=65325660, 已保存 -> .\pems11_2001.csv
2002: 节点数=631, 行数=66315576, 已保存 -> .\pems11_2002.csv
2003: 节点数=646, 行数=67892016, 已保存 -> .\pems11_2003.csv
2004: 节点数=646, 行数=67093464, 已保存 -> .\pems11_2004.csv
2005: 节点数=838, 行数=88001328, 已保存 -> .\pems11_2005.csv
2006: 节点数=944, 行数=98935008, 已保存 -> .\pems11_2006.csv
2007: 节点数=1065, 行数=111848640, 已保存 -> .\pems11_2007.csv
2008: 节点数=1098, 行数=113998860, 已保存 -> .\pems11_2008.csv
2009: 节点数=1136, 行数=118249164, 已保存 -> .\pems11_2009.csv
2010: 节点数=1193, 行数=124777476, 已保存 -> .\pems11_2010.csv
2011: 节点数=1221, 行数=127575300, 已保存 -> .\pems11_2011.csv
2012: 节点数=1273, 行数=128133780, 已保存 -> .\pems11_2012.csv


## 4. 各年节点数汇总

In [ ]:
summary = pd.DataFrame({
    'year': list(year_nodes.keys()),
    'anchor_date': [year_anchor[y] for y in year_nodes],
    'n_nodes': [len(year_nodes[y]) for y in year_nodes],
}).sort_values('year').reset_index(drop=True)
summary

## 5. Pivot 处理：行=5min时间步, 列=Station, 值=Total Flow

读取每年的 `pems11_{year}.csv`，pivot 成：
- **行**：完整的 5 分钟时间序列（缺失时间步自动补 NaN）
- **列**：每个 Station ID
- **值**：Total Flow

输出保存为 `pems11_{year}_flow.csv`

In [1]:
import os, re, glob
import pandas as pd
from datetime import date

FLOW_OUTPUT_DIR = '.'
FIRST_DATE = date(1999, 7, 1)
DISTRICT = 'pems11'

_csv_files = glob.glob(os.path.join(FLOW_OUTPUT_DIR, f'{DISTRICT}_[0-9][0-9][0-9][0-9].csv'))
years = sorted(int(re.search(rf'{DISTRICT}_(\d{{4}})\.csv', f).group(1)) for f in _csv_files)
print(f"扫描到 {len(years)} 个年度 CSV: {years[0]}~{years[-1]}")

def pivot_yearly_csv(year, first_date=FIRST_DATE, output_dir=FLOW_OUTPUT_DIR):
    src_path = os.path.join(output_dir, f'{DISTRICT}_{year}.csv')
    if not os.path.exists(src_path):
        print(f"  [跳过] {src_path} 不存在")
        return

    df = pd.read_csv(src_path, usecols=['Timestamp', 'Station', 'Total Flow'])
    print(f"  读取完成: {len(df):,} 行")

    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    pivot = df.pivot_table(
        index='Timestamp', columns='Station',
        values='Total Flow', aggfunc='sum'
    )
    del df

    pivot.columns = pivot.columns.astype(int)
    pivot = pivot[sorted(pivot.columns)]

    if year == first_date.year:
        start_dt = pd.Timestamp(first_date)
    else:
        start_dt = pd.Timestamp(year, 1, 1)
    end_dt = pd.Timestamp(year, 12, 31, 23, 55)

    full_idx = pd.date_range(start=start_dt, end=end_dt, freq='5min')
    pivot = pivot.reindex(full_idx)
    pivot.index.name = 'Timestamp'

    out_path = os.path.join(output_dir, f'{DISTRICT}_{year}_flow.csv')
    pivot.to_csv(out_path)

    n_nan_rows = pivot.isna().all(axis=1).sum()
    print(f"  {year}: stations={pivot.shape[1]}, "
          f"时间步={pivot.shape[0]}(期望{len(full_idx)}), "
          f"全NaN行={n_nan_rows}, 已保存 -> {out_path}")
    return pivot.shape

print("pivot_yearly_csv 函数已定义")

扫描到 15 个年度 CSV: 1999~2013
pivot_yearly_csv 函数已定义


In [2]:
import time

results = {}
for year in years:
    print(f"\n{'='*60}")
    print(f"处理 {year} 年 ...")
    t0 = time.time()
    shape = pivot_yearly_csv(year)
    elapsed = time.time() - t0
    print(f"  耗时: {elapsed:.1f}s")
    if shape:
        results[year] = shape

print(f"\n{'='*60}")
print(f"全部完成! 共处理 {len(results)} 年")


处理 1999 年 ...
  读取完成: 22,463,520 行
  1999: stations=424, 时间步=52992(期望52992), 全NaN行=12, 已保存 -> .\pems11_1999_flow.csv
  耗时: 139.1s

处理 2000 年 ...
  读取完成: 58,592,352 行
  2000: stations=556, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems11_2000_flow.csv
  耗时: 306.9s

处理 2001 年 ...
  读取完成: 65,325,660 行
  2001: stations=622, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems11_2001_flow.csv
  耗时: 502.9s

处理 2002 年 ...
  读取完成: 66,315,576 行
  2002: stations=631, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems11_2002_flow.csv
  耗时: 449.9s

处理 2003 年 ...
  读取完成: 67,892,016 行
  2003: stations=646, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems11_2003_flow.csv
  耗时: 466.8s

处理 2004 年 ...
  读取完成: 67,093,464 行
  2004: stations=646, 时间步=105408(期望105408), 全NaN行=24, 已保存 -> .\pems11_2004_flow.csv
  耗时: 504.4s

处理 2005 年 ...
  读取完成: 88,001,328 行
  2005: stations=838, 时间步=105120(期望105120), 全NaN行=24, 已保存 -> .\pems11_2005_flow.csv
  耗时: 740.9s

处理 2006 年 ...
  读取完成: 98,935,008 行
  2006: stations=944, 时间步=105120(期望

## 6. 检查输出结果汇总

In [3]:
summary_flow = pd.DataFrame([
    {'year': y, 'rows': s[0], 'stations': s[1]}
    for y, s in results.items()
])
summary_flow = summary_flow.sort_values('year').reset_index(drop=True)
print("各年 flow pivot 结果汇总:")
summary_flow

各年 flow pivot 结果汇总:


,year,rows,stations
0,1999,52992,424
1,2000,105408,556
2,2001,105120,622
3,2002,105120,631
4,2003,105120,646
5,2004,105408,646
6,2005,105120,838
7,2006,105120,944
8,2007,105120,1065
9,2008,105408,1098


In [4]:
check = pd.read_csv(f'./{DISTRICT}_{years[0]}_flow.csv', index_col=0, parse_dates=True, nrows=10)
print(f"{DISTRICT}_{years[0]}_flow.csv 前 10 行, shape={check.shape}")
check

pems11_1999_flow.csv 前 10 行, shape=(10, 424)


,1100270,1100307,1100310,1100313,1100326,1100330,1100348,1100353,1100369,1100372,...,1111509,1111510,1111511,1111513,1111562,1111564,1111572,1111573,1111574,1111578
Timestamp,,,,,,,,,,,,,,,,,,,,,
1999-07-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:35:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-01 00:40:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
